# Exercise 1 + 2 — Load AdventureWorks CSVs
Prerequisite (Exercise 1, done in the UI): catalog `adventure_works`, volume `raw_data` in schema `default`,
with the 6 CSVs uploaded into it. The CSVs are **tab-separated with no header row**; column names come from
the CREATE TABLE statements in Microsoft's `instawdb.sql`.

In [0]:
# Cell 1 — Column names and types from instawdb.sql
from pyspark.sql import functions as F

VOLUME = "/Volumes/adventure_works/default/raw_data"

SCHEMAS = {
    "SalesOrderDetail": [
        ("SalesOrderID", "int"), ("SalesOrderDetailID", "int"), ("CarrierTrackingNumber", "string"),
        ("OrderQty", "int"), ("ProductID", "int"), ("SpecialOfferID", "int"),
        ("UnitPrice", "decimal(19,4)"), ("UnitPriceDiscount", "decimal(19,4)"), ("LineTotal", "decimal(38,6)"),
        ("rowguid", "string"), ("ModifiedDate", "timestamp")],
    "Product": [
        ("ProductID", "int"), ("Name", "string"), ("ProductNumber", "string"), ("MakeFlag", "boolean"),
        ("FinishedGoodsFlag", "boolean"), ("Color", "string"), ("SafetyStockLevel", "int"), ("ReorderPoint", "int"),
        ("StandardCost", "decimal(19,4)"), ("ListPrice", "decimal(19,4)"), ("Size", "string"),
        ("SizeUnitMeasureCode", "string"), ("WeightUnitMeasureCode", "string"), ("Weight", "decimal(8,2)"),
        ("DaysToManufacture", "int"), ("ProductLine", "string"), ("Class", "string"), ("Style", "string"),
        ("ProductSubcategoryID", "int"), ("ProductModelID", "int"), ("SellStartDate", "timestamp"),
        ("SellEndDate", "timestamp"), ("DiscontinuedDate", "timestamp"), ("rowguid", "string"),
        ("ModifiedDate", "timestamp")],
    "SalesTerritory": [
        ("TerritoryID", "int"), ("Name", "string"), ("CountryRegionCode", "string"), ("Group", "string"),
        ("SalesYTD", "decimal(19,4)"), ("SalesLastYear", "decimal(19,4)"), ("CostYTD", "decimal(19,4)"),
        ("CostLastYear", "decimal(19,4)"), ("rowguid", "string"), ("ModifiedDate", "timestamp")],
    "PurchaseOrderDetail": [
        ("PurchaseOrderID", "int"), ("PurchaseOrderDetailID", "int"), ("DueDate", "timestamp"), ("OrderQty", "int"),
        ("ProductID", "int"), ("UnitPrice", "decimal(19,4)"), ("LineTotal", "decimal(19,4)"),
        ("ReceivedQty", "decimal(8,2)"), ("RejectedQty", "decimal(8,2)"), ("StockedQty", "decimal(9,2)"),
        ("ModifiedDate", "timestamp")],
    "PurchaseOrderHeader": [
        ("PurchaseOrderID", "int"), ("RevisionNumber", "int"), ("Status", "int"), ("EmployeeID", "int"),
        ("VendorID", "int"), ("ShipMethodID", "int"), ("OrderDate", "timestamp"), ("ShipDate", "timestamp"),
        ("SubTotal", "decimal(19,4)"), ("TaxAmt", "decimal(19,4)"), ("Freight", "decimal(19,4)"),
        ("TotalDue", "decimal(19,4)"), ("ModifiedDate", "timestamp")],
    "Vendor": [
        ("BusinessEntityID", "int"), ("AccountNumber", "string"), ("Name", "string"), ("CreditRating", "int"),
        ("PreferredVendorStatus", "boolean"), ("ActiveFlag", "boolean"), ("PurchasingWebServiceURL", "string"),
        ("ModifiedDate", "timestamp")],
}

def load(name):
    cols = SCHEMAS[name]
    df = (spark.read.option("sep", "\t").option("header", "false").option("encoding", "UTF-8")
          .csv(f"{VOLUME}/{name}.csv"))
    assert len(df.columns) == len(cols), f"{name}: CSV has {len(df.columns)} columns, expected {len(cols)}"
    df = df.toDF(*[c for c, _ in cols])
    return df.select([F.col(f"`{c}`").cast(t).alias(c) for c, t in cols])

In [0]:
# Cell 2 — Exercise 2: the star schema (fact_sales, dim_product, dim_sales_territory) in adventure_works.default
star = {"fact_sales": "SalesOrderDetail", "dim_product": "Product", "dim_sales_territory": "SalesTerritory"}
for table, src in star.items():
    load(src).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"adventure_works.default.{table}")
    print(f"adventure_works.default.{table}: {spark.table(f'adventure_works.default.{table}').count():,} rows")

adventure_works.default.fact_sales: 121,317 rows
adventure_works.default.dim_product: 504 rows
adventure_works.default.dim_sales_territory: 10 rows


In [0]:
# Cell 3 — Bronze layer for Lessons 3-4 (6 tables, the names the Bronze to Silver notebook expects)
spark.sql("CREATE SCHEMA IF NOT EXISTS adventure_works.bronze")
bronze = {
    "product": "Product", "purchase_order_detail": "PurchaseOrderDetail",
    "purchase_order_header": "PurchaseOrderHeader", "sales_order_detail": "SalesOrderDetail",
    "sales_territory": "SalesTerritory", "vendor": "Vendor",
}
for table, src in bronze.items():
    load(src).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"adventure_works.bronze.{table}")
    print(f"adventure_works.bronze.{table}: {spark.table(f'adventure_works.bronze.{table}').count():,} rows")

adventure_works.bronze.product: 504 rows
adventure_works.bronze.purchase_order_detail: 8,845 rows
adventure_works.bronze.purchase_order_header: 4,012 rows
adventure_works.bronze.sales_order_detail: 121,317 rows
adventure_works.bronze.sales_territory: 10 rows
adventure_works.bronze.vendor: 104 rows


In [0]:
# Cell 4 — Exercise 3 check: columns are named, not _c0, _c1...
display(spark.sql("DESCRIBE adventure_works.default.fact_sales"))

col_name,data_type,comment
SalesOrderID,int,null
SalesOrderDetailID,int,null
CarrierTrackingNumber,string,null
OrderQty,int,null
ProductID,int,null
SpecialOfferID,int,null
UnitPrice,"decimal(19,4)",null
UnitPriceDiscount,"decimal(19,4)",null
LineTotal,"decimal(38,6)",null
rowguid,string,null
